In [1]:
# load libraies

%run py_libraries.py

/Users/4476224/.local/lib/python3.8/site-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.6.0 and strictly below 2.9.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.13.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported configuration, either change the TensorFlow version or the TensorFlow Addons's version. 
You can find the compatibility matrix in TensorFlow Addon's readme:
https://github.com/tensorflow/addons
  warnings.warn(


In [2]:
# loading utility files

from utility.sv_fig import savefig
# from utility.mi_score import MI_score



In [3]:
# def savefig(filename, crop = True):
#     plt.savefig('{}.pdf'.format(filename))

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', None)

# load data
data = pd.read_csv('data/baselinedata_37F.csv')
# data = data.loc[0:317]

print(data.shape)

(184, 38)


In [5]:
data = data.apply(pd.to_numeric) # convert all columns of Ndata to numerics

In [6]:
# count number of NCa

# NCa
data[data.CACHEXSTAGE0VIG == 0].shape

(28, 38)

In [7]:
# count number of PCa

# PCa
data[data.CACHEXSTAGE0VIG == 1].shape

(53, 38)

In [8]:
# count number of Ca

# Ca
data[data.CACHEXSTAGE0VIG == 2].shape

(103, 38)

In [9]:
# PCa v Ca
ndata_PCa_Ca = data[data["CACHEXSTAGE0VIG"] != 0]
ndata_PCa_Ca.reset_index(drop=True, inplace=True)
print(ndata_PCa_Ca.shape)

(156, 38)


In [10]:
## PCa v Ca

# ndata_PCa_Ca = ndata_PCa_Ca.apply(pd.to_numeric) # convert all columns of Ndata to numerics

# # PCa
# # Replace all occurrences of 1 with 0
# ndata_PCa_Ca["CACHEXSTAGE0VIG"] = ndata_PCa_Ca["CACHEXSTAGE0VIG"].replace(1, 0)

# Ca
# Replace all occurrences of 2 with 0
ndata_PCa_Ca["CACHEXSTAGE0VIG"] = ndata_PCa_Ca["CACHEXSTAGE0VIG"].replace(2, 0)



In [11]:
# count number of PCa

# PCa
ndata_PCa_Ca[ndata_PCa_Ca.CACHEXSTAGE0VIG == 1].shape

(53, 38)

In [12]:
# count number of Ca

# Ca
ndata_PCa_Ca[ndata_PCa_Ca.CACHEXSTAGE0VIG == 0].shape

(103, 38)

In [13]:
# calculate proportion of missingness in ndata_noR

# total num of NaN in the ndata_noR
total_nan_count = ndata_PCa_Ca.isna().sum().sum()

# total num of cells in ndata_noR
total_cells = ndata_PCa_Ca.size

# proportion of NaN
NaN_proportion = total_nan_count / total_cells

print('NaN_proportions:',NaN_proportion)


NaN_proportions: 0.01720647773279352


In [14]:
# table 

col_tab = ['ENA.78', 'IFN.y', 'IL.10', 'IL.6', 'IL.8', 'MCP.1', 'MDC', 'MIP.1a', 'TNF.a', 'C.peptide', 'G.CSF', 'IL.22', 'Insulin', 
           'Leptin', 'MIP.3a', 'GRO.a', 'HGF', 'MMP.2', 'Adiponectin', 'CRP', 'GDF.15', 'TIMP.1', 'TGF.B2', 'TGF.B1', 'PPAR.y', 
           'HIF.1a', 'Laminin', 'HbA1c', 'CA19.9', 'Glucose', 'HDL', 'CCK', 'LDL', 'Triglyceride', 'Albumin', 'Lumican', 'ZAG', 'CACHEXSTAGE0VIG']

categorical = ['CACHEXSTAGE0VIG']

groupby = ['CACHEXSTAGE0VIG']

myTable_demo_lg = TableOne(ndata_PCa_Ca,columns=col_tab,categorical=categorical,groupby=groupby,pval=True)

print(myTable_demo_lg.tabulate(tablefmt="latex"))





\begin{tabular}{lllllll}
\hline
                         &     & Missing   & Overall    & 0.0         & 1.0        & P-Value   \\
\hline
 n                       &     &           & 156        & 103         & 53         &           \\
 ENA.78, mean (SD)       &     & 0         & 13.2 (1.2) & 13.2 (1.2)  & 13.2 (1.2) & 0.839     \\
 IFN.y, mean (SD)        &     & 0         & 6.6 (1.4)  & 6.7 (1.2)   & 6.2 (1.6)  & 0.027     \\
 IL.10, mean (SD)        &     & 1         & 2.8 (1.6)  & 3.1 (1.4)   & 2.2 (1.7)  & 0.001     \\
 IL.6, mean (SD)         &     & 0         & 5.1 (1.4)  & 5.2 (1.4)   & 4.9 (1.4)  & 0.224     \\
 IL.8, mean (SD)         &     & 0         & 8.1 (1.2)  & 8.2 (1.3)   & 7.8 (0.9)  & 0.031     \\
 MCP.1, mean (SD)        &     & 0         & 11.5 (0.6) & 11.5 (0.6)  & 11.5 (0.5) & 0.698     \\
 MDC, mean (SD)          &     & 0         & 13.6 (0.6) & 13.6 (0.6)  & 13.6 (0.5) & 0.747     \\
 MIP.1a, mean (SD)       &     & 5         & 7.7 (1.3)  & 7.9 (1.3)   & 7.3 (1.

In [15]:
# checking number of patients not missing data

ndata_PCa_Ca_noMiss = ndata_PCa_Ca.copy()
num_complete_rows = ndata_PCa_Ca_noMiss.dropna().shape[0]
print(num_complete_rows)

77


In [16]:
X = ndata_PCa_Ca.iloc[:,:-1]
y = ndata_PCa_Ca.iloc[:, -1] #.values

# X.head()


In [17]:
def data_split(X,y,rnd_st,tst_sz):
    X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                        test_size = tst_sz, 
                                                        random_state=rnd_st,
                                                        stratify=y)
    
    return X_train, X_test, y_train, y_test

In [18]:
#

X_train, X_test, y_train, y_test = data_split(X,y,rnd_st=1234,tst_sz=0.30)        # working best for now
# X_train, X_test, y_train, y_test = data_split(X,y,rnd_st=1234,tst_sz=0.20)        # working best for now


In [19]:
X_train

,ENA.78,IFN.y,IL.10,IL.6,IL.8,MCP.1,MDC,MIP.1a,TNF.a,C.peptide,G.CSF,IL.22,Insulin,Leptin,MIP.3a,GRO.a,HGF,MMP.2,Adiponectin,CRP,GDF.15,TIMP.1,TGF.B2,TGF.B1,PPAR.y,HIF.1a,Laminin,HbA1c,CA19.9,Glucose,HDL,CCK,LDL,Triglyceride,Albumin,Lumican,ZAG
5,11.571437,5.954002,3.360089,7.586015,7.403112,11.303186,13.427446,7.893635,5.566243,14.549587,7.305144,0.175595,5.688919,15.793603,6.894301,11.209964,9.691044,15.233731,23.211094,24.851405,9.724059,18.012469,2.986094,16.157996,0.349365,NaN,10.183840,7.161475,2.428678,6.226354,9.196226,8.367030,14.189518,4.186659,37.995395,20.722962,22.016700
29,14.420232,7.821295,0.956161,3.387694,7.306813,11.770866,13.680470,6.808157,4.565455,15.328903,6.296545,10.703387,6.974682,17.720672,10.307599,11.871803,9.726308,12.409409,24.866570,22.173123,10.244650,19.447295,6.711946,16.244782,2.756596,8.648598,10.773644,10.049132,6.793740,6.271687,8.000248,8.630937,14.261354,NaN,38.014731,20.533696,22.372208
102,13.215464,7.014452,5.937433,6.039912,10.876025,11.129401,12.694413,8.946007,7.215764,13.365601,5.878933,3.659251,3.598877,NaN,10.036192,12.103603,8.973863,14.338985,24.368986,22.382440,11.553215,19.609397,NaN,13.907156,NaN,9.155352,12.122637,8.538352,8.852573,7.353200,9.760327,6.573238,14.829946,6.003939,38.088328,19.914793,19.334578
1,13.413987,7.017272,4.406002,5.749313,8.325166,11.946972,13.423097,7.808320,5.721612,12.590742,7.745697,4.475445,4.410839,17.053247,8.528102,12.007775,8.767061,15.312314,23.890241,24.537325,10.664005,18.191906,5.379034,15.735355,-0.163268,8.651576,9.384041,8.889537,5.610730,6.782055,8.843007,8.326537,15.076799,4.097021,37.911887,21.853215,22.753599
57,12.638317,7.393547,3.212959,5.798775,9.727159,11.415929,14.319997,7.407946,5.363609,13.469339,6.499778,1.574101,4.597330,13.992584,8.112472,11.709870,9.613266,15.745228,25.327985,25.888680,12.108865,19.499541,7.967434,16.432319,1.149259,11.018477,11.388569,9.454949,1.503349,6.287251,9.452663,8.140078,14.385566,5.591949,38.034190,20.400621,20.931740
141,14.088707,5.886263,3.200297,5.041457,7.513796,11.312137,13.163043,6.828981,5.416804,14.825200,6.235098,-0.881875,6.872611,17.084725,NaN,12.684581,9.052307,14.605147,23.942239,24.950249,9.127613,16.853731,7.666549,16.710319,0.282143,8.439428,10.098526,9.251326,9.231272,6.682672,9.273376,6.672722,13.524657,6.122466,37.835880,21.519275,23.134963
146,14.627135,8.514043,3.889617,6.225673,7.673457,11.657319,13.979170,NaN,6.128145,14.153639,6.871834,3.217786,5.556573,16.605306,8.149309,13.457014,9.929510,14.952736,23.492572,18.477377,10.627265,18.071022,6.622817,16.389962,1.063503,8.891121,11.529287,9.565666,0.895303,6.952194,8.607012,10.163067,15.787631,4.379967,37.085002,22.093731,23.158510
140,14.128681,7.356263,3.111289,5.442122,7.240588,11.035531,12.963283,NaN,5.393742,12.997811,6.341781,1.698522,4.213408,16.680030,NaN,13.605435,9.097723,15.507175,24.740257,20.563417,11.215846,18.317887,7.038488,15.420729,1.456280,7.753899,10.720709,8.456371,1.462052,6.551070,9.169887,9.299957,14.919286,5.209336,38.288240,22.051176,22.685319
36,12.153687,4.448468,-0.664277,3.273246,10.389400,11.395320,13.238473,6.861536,4.278966,14.040429,6.858901,2.201897,5.467095,15.833681,4.992553,11.243959,8.767848,14.847976,24.800418,18.893435,10.159906,18.160504,6.933180,15.684262,2.344260,10.231209,10.898638,7.904767,7.829196,7.712486,7.771318,7.725871,13.550060,3.901398,38.280232,20.149858,21.297262
127,14.334683,7.740484,2.006972,4.499953,6.829247,11.760763,13.849800,6.278901,4.719279,13.641306,7.390207,1.985673,5.037164,17.213712,5.508997,11.795495,9.079641,11.907184,24.399992,23.541954,10.671907,18.612427,8.371205,16.290114,0.558757,11.374318,10.648889,9.295732,6.807819,6.538911,9.501949,8.745190,13.938047,4.941670,38.067689,22.048571,23.380936


In [20]:
print(X_train.shape)


(109, 37)


In [21]:
# checking number of patients not missing data

X_train_noMiss = X_train.copy()
num_complete_rows = X_train_noMiss.dropna().shape[0]
print(num_complete_rows)

51


In [22]:
# checking number of patients not missing data

X_test_noMiss = X_test.copy()
num_complete_rows = X_test_noMiss.dropna().shape[0]
print(num_complete_rows)

26


In [23]:
# scaler0 = StandardScaler().fit(X_train) # build a scaler for the training data
from sklearn.preprocessing import MinMaxScaler, RobustScaler, MaxAbsScaler

# scaler0 = MinMaxScaler().fit(X_train) # build a scaler for the training data
scaler0 = MaxAbsScaler().fit(X_train) # build a scaler for the training data

In [24]:
# scaled x_train

X_train_sc = scaler0.transform(X_train) # use the scaler to transform the training data


In [25]:
# scaled x_test

X_test_sc = scaler0.transform(X_test) # use the scaler to transform the training data


In [26]:
print(X_train_sc[:,0])

[0.74465043 0.9279774  0.85044761 0.86322308 0.81330682 0.90664295
 0.94129212 0.9092154  0.78211968 0.92247213 0.97570834 0.85745951
 0.85594323 0.86776194 0.9318794  0.75646898 0.91050922 0.93337572
 0.71479814 0.81635494 0.93716597 0.8403565  0.64701394 0.83461554
 0.68230132 0.83810913 0.98599284 0.86265292 0.78660584 0.87790934
 0.72496832 0.85116042 0.84589175 0.81606155 0.94194464 0.84434362
 0.85024779 0.80044241 0.7769102  0.78941051 0.79660622 0.98551965
 1.         0.86550362 0.88704721 0.85426687 0.99212991 0.87198411
 0.84049278 0.88148344 0.98914424 0.78244313 0.89609432 0.86289877
 0.85017471 0.72617575 0.84907762 0.89381914 0.80729936 0.88971786
 0.80185408 0.88169823 0.81038525 0.90181687 0.82370608 0.97098998
 0.86744124 0.83056891 0.85289602 0.82483231 0.90139302 0.81054629
 0.86474727 0.93903269 0.74775259 0.91710366 0.89736318 0.90267784
 0.83779233 0.7875807  0.81908977 0.89802358 0.74768773 0.93551295
 0.96407395 0.74287755 0.59469708 0.94487653 0.79442008 0.8463

In [27]:
X_train_sc.shape

(109, 37)

In [28]:
colnames=X_train.columns.tolist()

# colnames

In [29]:
# X_train

# Conversion with custom column names
X_train_sc_pd = pd.DataFrame(X_train_sc, columns=colnames)


In [30]:
# save to csv file

# 
X_train_sc_pd.to_csv('data/norm_X_train_PCa_Ca.csv', index=False)

In [31]:
# X_test

# Conversion with custom column names
X_test_sc_pd = pd.DataFrame(X_test_sc, columns=colnames)


In [32]:
# save to csv file

# 
X_test_sc_pd.to_csv('data/norm_X_test_PCa_Ca.csv', index=False)

In [33]:
# save to csv file

# 
y_train.to_csv('data/y_train_PCa_Ca.csv', index=False)

In [34]:
# save to csv file

# 
y_test.to_csv('data/y_test_PCa_Ca.csv', index=False)